# Countries, Entities, and groups
This Jupyter notebook will elaborate on the creation of the dataset

## Chapter 1: Our World in Data

First we want to create a dataset of all countries, their ISO codes and regions from Our World in Data. Our main sources would be the OWID's [World map region definitions](https://ourworldindata.org/world-region-map-definitions) page and OWID's [standard entity names](https://github.com/owid/energy-data/tree/master/scripts/input/shared). ([source](https://raw.githubusercontent.com/owid/energy-data/master/scripts/input/shared/iso_codes.csv)). 

However, these sources did not list all available OWID's entities, so we will also interpolate our dataset with entities listed in OWID's [Energy Data](https://github.com/owid/energy-data) ([source](https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv)), and [CO2 Data](https://github.com/owid/co2-data) ([source](https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv))

### World map region definitions

First, we manually download the datasets from https://ourworldindata.org/world-region-map-definitions and stored them in the "owid" directory

Then, we read those downloaded csv file, store each csv table in a DataFrame, and merge the DataFrames together using the Code attribute

In [1]:
# import the needed packages
import pandas as pd
import os

In [2]:
# read the files

datafiles = []

for dirpath, _, filenames in os.walk("owid"):
    for f in filenames:
        datafiles.append(os.path.join(dirpath, f))

datafiles

['owid\\continents-according-to-our-world-in-data.csv',
 'owid\\who-regions.csv',
 'owid\\world-regions-according-to-the-world-bank.csv',
 'owid\\world-regions-sdg-united-nations.csv']

In [3]:
# load each file into a DataFrame

dfs = []

for f in datafiles:
    # drop the Year column since we don't need it
    df = pd.read_csv(f).drop('Year', axis=1)
    dfs.append(df)

dfs

[                    Entity      Code Continent
 0                 Abkhazia  OWID_ABK      Asia
 1              Afghanistan       AFG      Asia
 2    Akrotiri and Dhekelia  OWID_AKD      Asia
 3                  Albania       ALB    Europe
 4                  Algeria       DZA    Africa
 ..                     ...       ...       ...
 280             Yugoslavia  OWID_YGS    Europe
 281                 Zambia       ZMB    Africa
 282               Zanzibar  OWID_ZAN    Africa
 283               Zimbabwe       ZWE    Africa
 284          Åland Islands       ALA    Europe
 
 [285 rows x 3 columns],
           Entity Code             WHO region
 0    Afghanistan  AFG  Eastern Mediterranean
 1        Albania  ALB                 Europe
 2        Algeria  DZA                 Africa
 3        Andorra  AND                 Europe
 4         Angola  AGO                 Africa
 ..           ...  ...                    ...
 189    Venezuela  VEN               Americas
 190      Vietnam  VNM       

In [4]:
# merge all dataframes based on Entity and Code

df_merged = dfs[0]

for df in dfs[1:]:
    df_merged = pd.merge(df_merged, df, on=["Entity", "Code"], how="outer")

df_merged

,Entity,Code,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations
0,Abkhazia,OWID_ABK,Asia,NaN,NaN,NaN
1,Afghanistan,AFG,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia
2,Akrotiri and Dhekelia,OWID_AKD,Asia,NaN,NaN,NaN
3,Albania,ALB,Europe,Europe,Europe and Central Asia,Europe and Northern America
4,Algeria,DZA,Africa,Africa,Middle East and North Africa,Northern Africa and Western Asia
...,...,...,...,...,...,...
283,Zimbabwe,ZWE,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa
284,Åland Islands,ALA,Europe,NaN,NaN,NaN
285,Micronesia,NaN,NaN,NaN,East Asia and Pacific,Oceania
286,Aland Islands,NaN,NaN,NaN,NaN,Europe and Northern America


### OWID Energy Data

In [5]:
# read the energy data file
df = pd.read_csv("https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv")
df


,iso_code,country,year,coal_prod_change_pct,coal_prod_change_twh,gas_prod_change_pct,gas_prod_change_twh,oil_prod_change_pct,oil_prod_change_twh,energy_cons_change_pct,...,solar_consumption,solar_elec_per_capita,solar_energy_per_capita,wind_share_elec,wind_cons_change_pct,wind_share_energy,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_energy_per_capita
0,AFG,Afghanistan,1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,1901,NaN,0.000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,1902,NaN,0.000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Afghanistan,1903,NaN,0.000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Afghanistan,1904,NaN,0.000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17234,ZWE,Zimbabwe,2016,-37.694,-12.257,NaN,0.0,NaN,0.0,-14.611,...,NaN,0.713,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
17235,ZWE,Zimbabwe,2017,8.375,1.697,NaN,NaN,NaN,NaN,-1.564,...,NaN,0.702,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
17236,ZWE,Zimbabwe,2018,14.336,3.148,NaN,NaN,NaN,NaN,3.409,...,NaN,0.693,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
17237,ZWE,Zimbabwe,2019,-21.529,-5.405,NaN,NaN,NaN,NaN,4.052,...,NaN,0.683,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN


In [6]:

# retrieve all iso codes and countries available in the datafile
df = df[["iso_code", "country"]].drop_duplicates()
df

,iso_code,country
0,AFG,Afghanistan
121,OWID_AFR,Africa
242,ALB,Albania
363,DZA,Algeria
484,ASM,American Samoa
...,...,...
16732,OWID_WRL,World
16854,YEM,Yemen
16905,NaN,Yugoslavia
16997,ZMB,Zambia


In [7]:
# set index as iso code and country
df_owid_energy_data = df.set_index(["iso_code", "country"])
df_owid_energy_data


,
iso_code,country
AFG,Afghanistan
OWID_AFR,Africa
ALB,Albania
DZA,Algeria
ASM,American Samoa
...,...
OWID_WRL,World
YEM,Yemen
NaN,Yugoslavia


### OWID CO2 Data

In [8]:

# read the OWID CO2 data
df = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")
df


,iso_code,country,year,co2,co2_per_capita,trade_co2,cement_co2,cement_co2_per_capita,coal_co2,coal_co2_per_capita,...,ghg_excluding_lucf_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,population,gdp,primary_energy_consumption,energy_per_capita,energy_per_gdp
0,AFG,Afghanistan,1949,0.015,0.002,NaN,NaN,NaN,0.015,0.002,...,NaN,NaN,NaN,NaN,NaN,7624058.0,NaN,NaN,NaN,NaN
1,AFG,Afghanistan,1950,0.084,0.011,NaN,NaN,NaN,0.021,0.003,...,NaN,NaN,NaN,NaN,NaN,7752117.0,9.421400e+09,NaN,NaN,NaN
2,AFG,Afghanistan,1951,0.092,0.012,NaN,NaN,NaN,0.026,0.003,...,NaN,NaN,NaN,NaN,NaN,7840151.0,9.692280e+09,NaN,NaN,NaN
3,AFG,Afghanistan,1952,0.092,0.012,NaN,NaN,NaN,0.032,0.004,...,NaN,NaN,NaN,NaN,NaN,7935996.0,1.001733e+10,NaN,NaN,NaN
4,AFG,Afghanistan,1953,0.106,0.013,NaN,NaN,NaN,0.038,0.005,...,NaN,NaN,NaN,NaN,NaN,8039684.0,1.063052e+10,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25984,ZWE,Zimbabwe,2016,10.738,0.765,1.415,0.639,0.046,6.959,0.496,...,2.076,11.50,0.820,6.21,0.443,14030338.0,2.096179e+10,46.666,3326.073,2.226
25985,ZWE,Zimbabwe,2017,9.582,0.673,1.666,0.678,0.048,5.665,0.398,...,2.023,11.62,0.816,6.35,0.446,14236599.0,2.194784e+10,45.936,3226.617,2.093
25986,ZWE,Zimbabwe,2018,11.854,0.821,1.308,0.697,0.048,7.101,0.492,...,2.173,11.96,0.828,6.59,0.456,14438812.0,2.271535e+10,47.502,3289.887,2.091
25987,ZWE,Zimbabwe,2019,10.949,0.748,1.473,0.697,0.048,6.020,0.411,...,NaN,NaN,NaN,NaN,NaN,14645473.0,NaN,49.427,3374.877,NaN


In [9]:

# retrieve all iso codes and countries available in the datafile
df = df[["iso_code", "country"]].drop_duplicates()
df


,iso_code,country
0,AFG,Afghanistan
72,NaN,Africa
209,ALB,Albania
297,DZA,Algeria
402,AND,Andorra
...,...,...
25438,NaN,Wallis and Futuna
25458,OWID_WRL,World
25729,YEM,Yemen
25800,ZMB,Zambia


In [10]:

# set index as iso code and country
df_owid_co2_data = df.set_index(["iso_code", "country"])
df_owid_co2_data


,
iso_code,country
AFG,Afghanistan
NaN,Africa
ALB,Albania
DZA,Algeria
AND,Andorra
...,...
NaN,Wallis and Futuna
OWID_WRL,World
YEM,Yemen


In [11]:

# merge two country lists from energy data and co2 data
df_owid_data = pd.merge(df_owid_energy_data, df_owid_co2_data, how="outer", left_index=True, right_index=True)
df_owid_data


Empty DataFrame
Columns: []
Index: [(ABW, Aruba), (AFG, Afghanistan), (AGO, Angola), (AIA, Anguilla), (ALB, Albania), (AND, Andorra), (ANT, Netherlands Antilles), (ARE, United Arab Emirates), (ARG, Argentina), (ARM, Armenia), (ASM, American Samoa), (ATG, Antigua and Barbuda), (AUS, Australia), (AUT, Austria), (AZE, Azerbaijan), (BDI, Burundi), (BEL, Belgium), (BEN, Benin), (BES, Bonaire Sint Eustatius and Saba), (BFA, Burkina Faso), (BGD, Bangladesh), (BGR, Bulgaria), (BHR, Bahrain), (BHS, Bahamas), (BIH, Bosnia and Herzegovina), (BLR, Belarus), (BLZ, Belize), (BMU, Bermuda), (BOL, Bolivia), (BRA, Brazil), (BRB, Barbados), (BRN, Brunei), (BTN, Bhutan), (BWA, Botswana), (CAF, Central African Republic), (CAN, Canada), (CHE, Switzerland), (CHL, Chile), (CHN, China), (CIV, Cote d'Ivoire), (CMR, Cameroon), (COD, Democratic Republic of Congo), (COG, Congo), (COK, Cook Islands), (COL, Colombia), (COM, Comoros), (CPV, Cape Verde), (CRI, Costa Rica), (CUB, Cuba), (CUW, Curacao), (CXR, Christmas Island), (CYM, Cayman Islands), (CYP, Cyprus), (CZE, Czechia), (DEU, Germany), (DJI, Djibouti), (DMA, Dominica), (DNK, Denmark), (DOM, Dominican Republic), (DZA, Algeria), (ECU, Ecuador), (EGY, Egypt), (ERI, Eritrea), (ESH, Western Sahara), (ESP, Spain), (EST, Estonia), (ETH, Ethiopia), (FIN, Finland), (FJI, Fiji), (FLK, Falkland Islands), (FRA, France), (FRO, Faeroe Islands), (FSM, Micronesia (country)), (GAB, Gabon), (GBR, United Kingdom), (GEO, Georgia), (GHA, Ghana), (GIN, Guinea), (GLP, Guadeloupe), (GMB, Gambia), (GNB, Guinea-Bissau), (GNQ, Equatorial Guinea), (GRC, Greece), (GRD, Grenada), (GRL, Greenland), (GTM, Guatemala), (GUF, French Guiana), (GUM, Guam), (GUY, Guyana), (HKG, Hong Kong), (HND, Honduras), (HRV, Croatia), (HTI, Haiti), (HUN, Hungary), (IDN, Indonesia), (IND, India), (IRL, Ireland), (IRN, Iran), (IRQ, Iraq), (ISL, Iceland), ...]

[283 rows x 0 columns]

### Standard Entity Names

In [12]:

# read the standard entity names file
df = pd.read_csv("https://raw.githubusercontent.com/owid/energy-data/master/scripts/input/shared/iso_codes.csv")
df


,iso_code,Country
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands
...,...,...
260,ZAF,South Africa
261,ZMB,Zambia
262,ZWE,Zimbabwe
263,OWID_WRL,World


In [13]:

# drop duplicates
df.drop_duplicates(subset="iso_code")
df


,iso_code,Country
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands
...,...,...
260,ZAF,South Africa
261,ZMB,Zambia
262,ZWE,Zimbabwe
263,OWID_WRL,World


In [14]:
# rename column
df.rename(columns={'Country': 'country'}, inplace=True)
df


C:\Users\An\AppData\Local\Temp\ipykernel_13728\897098064.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'Country': 'country'}, inplace=True)


,iso_code,country
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands
...,...,...
260,ZAF,South Africa
261,ZMB,Zambia
262,ZWE,Zimbabwe
263,OWID_WRL,World


In [15]:

# set index as iso code and country
df_owid_country = df.set_index(["iso_code", "country"])
df_owid_country


,
iso_code,country
ABW,Aruba
AFG,Afghanistan
AGO,Angola
AIA,Anguilla
ALA,Åland Islands
...,...
ZAF,South Africa
ZMB,Zambia
ZWE,Zimbabwe


In [16]:
# merge the standard country dataframe with the owid energy+co2 dataframe
df_owid = pd.merge(df_owid_data, df_owid_country, how="outer",
                    left_index=True, right_index=True)
df_owid

Empty DataFrame
Columns: []
Index: [(ABW, Aruba), (AFG, Afghanistan), (AGO, Angola), (AIA, Anguilla), (ALA, Åland Islands), (ALB, Albania), (AND, Andorra), (ANT, Netherlands Antilles), (ARE, United Arab Emirates), (ARG, Argentina), (ARM, Armenia), (ASM, American Samoa), (ATF, French Southern Territories), (ATG, Antigua and Barbuda), (AUS, Australia), (AUT, Austria), (AZE, Azerbaijan), (BDI, Burundi), (BEL, Belgium), (BEN, Benin), (BES, Bonaire Sint Eustatius and Saba), (BFA, Burkina Faso), (BGD, Bangladesh), (BGR, Bulgaria), (BHR, Bahrain), (BHS, Bahamas), (BIH, Bosnia and Herzegovina), (BLM, Saint Barthélemy), (BLR, Belarus), (BLZ, Belize), (BMU, Bermuda), (BOL, Bolivia), (BRA, Brazil), (BRB, Barbados), (BRN, Brunei), (BRN, Brunei Darussalam), (BTN, Bhutan), (BVT, Bouvet Island), (BWA, Botswana), (CAF, Central African Republic), (CAN, Canada), (CCK, Cocos (Keeling) Islands), (CHE, Switzerland), (CHL, Chile), (CHN, China), (CIV, Cote d'Ivoire), (CMR, Cameroon), (COD, Democratic Republic of Congo), (COG, Congo), (COK, Cook Islands), (COL, Colombia), (COM, Comoros), (CPV, Cape Verde), (CRI, Costa Rica), (CUB, Cuba), (CUW, Curacao), (CXR, Christmas Island), (CYM, Cayman Islands), (CYP, Cyprus), (CZE, Czechia), (DEU, Germany), (DJI, Djibouti), (DMA, Dominica), (DNK, Denmark), (DOM, Dominican Republic), (DZA, Algeria), (ECU, Ecuador), (EGY, Egypt), (ERI, Eritrea), (ESH, Western Sahara), (ESP, Spain), (EST, Estonia), (ETH, Ethiopia), (FIN, Finland), (FJI, Fiji), (FLK, Falkland Islands), (FLK, Falkland Islands (Malvinas)), (FRA, France), (FRO, Faeroe Islands), (FRO, Faroe Islands), (FSM, Federated States of Micronesia), (FSM, Micronesia (country)), (GAB, Gabon), (GBR, United Kingdom), (GEO, Georgia), (GGY, Guernsey), (GHA, Ghana), (GIB, Gibraltar), (GIN, Guinea), (GLP, Guadeloupe), (GMB, Gambia), (GNB, Guinea-Bissau), (GNQ, Equatorial Guinea), (GRC, Greece), (GRD, Grenada), (GRL, Greenland), (GTM, Guatemala), (GUF, French Guiana), (GUM, Guam), (GUY, Guyana), ...]

[321 rows x 0 columns]

In [17]:
# reset index
df_owid = df_owid.reset_index()
df_owid


,iso_code,country
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands
...,...,...
316,NaN,Wake Island
317,NaN,Wallis and Futuna
318,NaN,West Germany
319,NaN,Western Africa


In [18]:
# rename columns to merge with df_merge
df_owid.rename(columns={'country': 'Entity', 'iso_code': 'Code'}, inplace=True)
df_owid

,Code,Entity
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,AIA,Anguilla
4,ALA,Åland Islands
...,...,...
316,NaN,Wake Island
317,NaN,Wallis and Futuna
318,NaN,West Germany
319,NaN,Western Africa


In [19]:
# merge with df_merged
df = pd.merge(df_owid, df_merged, how="outer", on=["Entity", "Code"])
df

,Code,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations
0,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean
1,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia
2,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa
3,AIA,Anguilla,North America,NaN,NaN,Latin America and Caribbean
4,ALA,Åland Islands,Europe,NaN,NaN,NaN
...,...,...,...,...,...,...
356,OWID_YGS,Yugoslavia,Europe,NaN,NaN,NaN
357,OWID_ZAN,Zanzibar,Africa,NaN,NaN,NaN
358,NaN,Micronesia,NaN,NaN,East Asia and Pacific,Oceania
359,NaN,Aland Islands,NaN,NaN,NaN,Europe and Northern America


### Process with Excel

In [20]:
# output to csv file to process with Excel
df.to_csv("output/owid.csv")
df

,Code,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations
0,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean
1,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia
2,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa
3,AIA,Anguilla,North America,NaN,NaN,Latin America and Caribbean
4,ALA,Åland Islands,Europe,NaN,NaN,NaN
...,...,...,...,...,...,...
356,OWID_YGS,Yugoslavia,Europe,NaN,NaN,NaN
357,OWID_ZAN,Zanzibar,Africa,NaN,NaN,NaN
358,NaN,Micronesia,NaN,NaN,East Asia and Pacific,Oceania
359,NaN,Aland Islands,NaN,NaN,NaN,Europe and Northern America


In the Excel file, we selected any countries without a Code and check if the country is duplicated in the file or not.
We also added all empty and NaN ISO code values with a specific id, in order not to merge NaN id values with df_wb in the future.
The cleaning process with Excel was done in the [custom/ProcessBook.xlsx](/custom/ProcessBook.xlsx) file
After processing with Excel, we reload the file back to continue.

In [21]:
# read the processed data from Excel
df_owid = pd.read_csv("custom/owid.csv")
df_owid

,Code,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations
0,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean
1,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia
2,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa
3,AIA,Anguilla,North America,NaN,NaN,Latin America and Caribbean
4,ALA,Aland Islands,Europe,NaN,NaN,Europe and Northern America
...,...,...,...,...,...,...
318,_empty_30,U.S. Pacific Islands,NaN,NaN,NaN,NaN
319,_empty_31,U.S. Territories,NaN,NaN,NaN,NaN
320,_empty_32,Upper-middle-income countries,NaN,NaN,NaN,NaN
321,_empty_33,Wake Island,NaN,NaN,NaN,NaN


## Chapter 2: World Bank

In [22]:
# import python package
import world_bank_data as wb


In [23]:
# get countries available in World Bank database
df_wb = wb.get_countries()
df_wb

,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
id,,,,,,,,,
ABW,AW,Aruba,Latin America & Caribbean,,High income,Not classified,Oranjestad,-70.0167,12.5167
AFE,ZH,Africa Eastern and Southern,Aggregates,,Aggregates,Aggregates,,NaN,NaN
AFG,AF,Afghanistan,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.5228
AFR,A9,Africa,Aggregates,,Aggregates,Aggregates,,NaN,NaN
AFW,ZI,Africa Western and Central,Aggregates,,Aggregates,Aggregates,,NaN,NaN
...,...,...,...,...,...,...,...,...,...
XZN,A5,Sub-Saharan Africa excluding South Africa and ...,Aggregates,,Aggregates,Aggregates,,NaN,NaN
YEM,YE,"Yemen, Rep.",Middle East & North Africa,Middle East & North Africa (excluding high inc...,Low income,IDA,Sana'a,44.2075,15.3520
ZAF,ZA,South Africa,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Upper middle income,IBRD,Pretoria,28.1871,-25.7460


### Merge World Bank data with OWID datas 

In [24]:
# save the World Bank's ISO code to another column before merging with df_owid
df_wb["WB"] = df_wb.index
df_wb

,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude,WB
id,,,,,,,,,,
ABW,AW,Aruba,Latin America & Caribbean,,High income,Not classified,Oranjestad,-70.0167,12.5167,ABW
AFE,ZH,Africa Eastern and Southern,Aggregates,,Aggregates,Aggregates,,NaN,NaN,AFE
AFG,AF,Afghanistan,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.5228,AFG
AFR,A9,Africa,Aggregates,,Aggregates,Aggregates,,NaN,NaN,AFR
AFW,ZI,Africa Western and Central,Aggregates,,Aggregates,Aggregates,,NaN,NaN,AFW
...,...,...,...,...,...,...,...,...,...,...
XZN,A5,Sub-Saharan Africa excluding South Africa and ...,Aggregates,,Aggregates,Aggregates,,NaN,NaN,XZN
YEM,YE,"Yemen, Rep.",Middle East & North Africa,Middle East & North Africa (excluding high inc...,Low income,IDA,Sana'a,44.2075,15.3520,YEM
ZAF,ZA,South Africa,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Upper middle income,IBRD,Pretoria,28.1871,-25.7460,ZAF


In [25]:
# Save the Code in the OWID column before merging with df_wb
df_owid["OWID"] = df_owid["Code"]
df_owid

,Code,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations,OWID
0,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW
1,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG
2,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO
3,AIA,Anguilla,North America,NaN,NaN,Latin America and Caribbean,AIA
4,ALA,Aland Islands,Europe,NaN,NaN,Europe and Northern America,ALA
...,...,...,...,...,...,...,...
318,_empty_30,U.S. Pacific Islands,NaN,NaN,NaN,NaN,_empty_30
319,_empty_31,U.S. Territories,NaN,NaN,NaN,NaN,_empty_31
320,_empty_32,Upper-middle-income countries,NaN,NaN,NaN,NaN,_empty_32
321,_empty_33,Wake Island,NaN,NaN,NaN,NaN,_empty_33


In [26]:
# merge df_owid with df_wb and write the table to a csv file to process with Excel
df = pd.merge(df_owid, df_wb, how="outer", left_on="Code", right_index=True)
df.to_csv("output/owidwb.csv")
df
# after this, I have opened the file in excel and clean the data manually

,Code,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations,OWID,iso2Code,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude,WB
0.0,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW,AW,Aruba,Latin America & Caribbean,,High income,Not classified,Oranjestad,-70.0167,12.51670,ABW
1.0,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG,AF,Afghanistan,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.52280,AFG
2.0,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO,AO,Angola,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Lower middle income,IBRD,Luanda,13.2420,-8.81155,AGO
3.0,AIA,Anguilla,North America,NaN,NaN,Latin America and Caribbean,AIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4.0,ALA,Aland Islands,Europe,NaN,NaN,Europe and Northern America,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
NaN,TSS,NaN,NaN,NaN,NaN,NaN,NaN,T6,Sub-Saharan Africa (IDA & IBRD countries),Aggregates,,Aggregates,Aggregates,,NaN,NaN,TSS
NaN,UMC,NaN,NaN,NaN,NaN,NaN,NaN,XT,Upper middle income,Aggregates,,Aggregates,Aggregates,,NaN,NaN,UMC
NaN,WLD,NaN,NaN,NaN,NaN,NaN,NaN,1W,World,Aggregates,,Aggregates,Aggregates,,NaN,NaN,WLD
NaN,XKX,NaN,NaN,NaN,NaN,NaN,NaN,XK,Kosovo,Europe & Central Asia,Europe & Central Asia (excluding high income),Upper middle income,IDA,Pristina,20.9260,42.56500,XKX


In the Excel file, we manually checked if there was any conflict between the World Bank dataset and the OWID dataset, and we also check if there were any duplicate countries.
The cleaning process with Excel was done in the [custom/ProcessBook.xlsx](/custom/ProcessBook.xlsx) file.
After finished checking and fixing the datasets, we reloaded the file back to continue.

In [27]:
df_owid_wb = pd.read_csv("custom/owid_wb.csv")
df_owid_wb

,Code,Entity Name,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations,OWID,WB,iso2Code3,name,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude
0,ABW,Aruba,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW,ABW,AW,Aruba,Latin America & Caribbean,NaN,High income,Not classified,Oranjestad,-70.0167,12.51670
1,AFG,Afghanistan,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG,AFG,AF,Afghanistan,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.52280
2,AGO,Angola,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO,AGO,AO,Angola,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Lower middle income,IBRD,Luanda,13.2420,-8.81155
3,AIA,Anguilla,Anguilla,North America,NaN,NaN,Latin America and Caribbean,AIA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ALA,Aland Islands,Aland Islands,Europe,NaN,NaN,Europe and Northern America,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,NaN,St. Kitts-Nevis-Anguilla,St. Kitts-Nevis-Anguilla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
392,NaN,U.S. Pacific Islands,U.S. Pacific Islands,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
393,NaN,U.S. Territories,U.S. Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,NaN,Wake Island,Wake Island,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Chapter 3: IMF's World Economic Outlook

In [28]:
# import the Python client
import weo


In [29]:
# read weo dataset
path, url = weo.download(2022, 1, filename="output/weo_2022_1.csv")

path
url

Already downloaded 2022-Apr WEO dataset at output\weo_2022_1.csv


'https://www.imf.org/-/media/Files/Publications/WEO/WEO-Database/2022/WEOApr2022all.ashx'

In [30]:
#  read all countries available in the WEO dataset 
df_weo: pd.DataFrame = weo.WEO(path).countries()
df_weo

,WEO Country Code,ISO,Country
0,512,AFG,Afghanistan
44,914,ALB,Albania
88,612,DZA,Algeria
132,171,AND,Andorra
176,614,AGO,Angola
...,...,...,...
8404,582,VNM,Vietnam
8448,487,WBG,West Bank and Gaza
8492,474,YEM,Yemen
8536,754,ZMB,Zambia


In [31]:
# select only the ISO column and Country column
df_weo = df_weo[["ISO", "Country"]]
df_weo

,ISO,Country
0,AFG,Afghanistan
44,ALB,Albania
88,DZA,Algeria
132,AND,Andorra
176,AGO,Angola
...,...,...
8404,VNM,Vietnam
8448,WBG,West Bank and Gaza
8492,YEM,Yemen
8536,ZMB,Zambia


### Merge with WB+OWID dataset

In [32]:
# store the ISO column to the WEO column to prepare for merging with WB+OWID dataset 
df_weo["WEO"] = df_weo["ISO"]
df_weo

,ISO,Country,WEO
0,AFG,Afghanistan,AFG
44,ALB,Albania,ALB
88,DZA,Algeria,DZA
132,AND,Andorra,AND
176,AGO,Angola,AGO
...,...,...,...
8404,VNM,Vietnam,VNM
8448,WBG,West Bank and Gaza,WBG
8492,YEM,Yemen,YEM
8536,ZMB,Zambia,ZMB


In [33]:
# merge df_weo with df_owid_wb
df = pd.merge(df_owid_wb, df_weo, left_on="Code", right_on="ISO", how="outer")
df

,Code,Entity Name,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations,OWID,WB,iso2Code3,...,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude,ISO,Country,WEO
0,ABW,Aruba,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW,ABW,AW,...,Latin America & Caribbean,NaN,High income,Not classified,Oranjestad,-70.0167,12.51670,ABW,Aruba,ABW
1,AFG,Afghanistan,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG,AFG,AF,...,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.52280,AFG,Afghanistan,AFG
2,AGO,Angola,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO,AGO,AO,...,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Lower middle income,IBRD,Luanda,13.2420,-8.81155,AGO,Angola,AGO
3,AIA,Anguilla,Anguilla,North America,NaN,NaN,Latin America and Caribbean,AIA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ALA,Aland Islands,Aland Islands,Europe,NaN,NaN,Europe and Northern America,ALA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,NaN,U.S. Territories,U.S. Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,NaN,Wake Island,Wake Island,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395,NaN,Western Africa,Western Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UVK,Kosovo,UVK


In [34]:
# write the table back to Excel to continue processing
df.to_csv("output/owid_wb_weo.csv")
df
# after this, I have opened the file in excel and clean the data manually

,Code,Entity Name,Entity,Continent,WHO region,World Region according to the World Bank,world-regions-according-to-the-united-nations,OWID,WB,iso2Code3,...,region,adminregion,incomeLevel,lendingType,capitalCity,longitude,latitude,ISO,Country,WEO
0,ABW,Aruba,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW,ABW,AW,...,Latin America & Caribbean,NaN,High income,Not classified,Oranjestad,-70.0167,12.51670,ABW,Aruba,ABW
1,AFG,Afghanistan,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG,AFG,AF,...,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.52280,AFG,Afghanistan,AFG
2,AGO,Angola,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO,AGO,AO,...,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Lower middle income,IBRD,Luanda,13.2420,-8.81155,AGO,Angola,AGO
3,AIA,Anguilla,Anguilla,North America,NaN,NaN,Latin America and Caribbean,AIA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ALA,Aland Islands,Aland Islands,Europe,NaN,NaN,Europe and Northern America,ALA,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,NaN,U.S. Territories,U.S. Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,NaN,Wake Island,Wake Island,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395,NaN,Western Africa,Western Africa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UVK,Kosovo,UVK


In the Excel file, we checked for conflicts between WEO, WB, and OWID, and check if there were any duplicate countries in the file.
The cleaning process with Excel was done in the [custom/ProcessBook.xlsx](/custom/ProcessBook.xlsx) file.
After finished checking and fixing the datasets, we put the final processed dataset in the entities sheet.
We reload the file back to finalize.

## Final
Finally, we moved the final dataset in the "entities" Excel sheet to the front, stored it in a csv.

In [35]:
# load the sheet back to df, and write the data to entities.csv
df = pd.read_excel("custom/ProcessBook.xlsx", sheet_name="entities")
df.to_csv("entities.csv", index=False)
df

,Code,Entity,OWID,OWID_Name,OWID_Continent,OWID_WHO_Region,OWID_WB_Region,OWID_UN_Region,WB,WB_ISO2,WB_Name,WB_region,WB_adminregion,WB_incomeLevel,WB_lendingType,WB_capitalCity,WB_longitude,WB_latitude,WEO,WEO_Country
0,ABW,Aruba,ABW,Aruba,North America,NaN,Latin America and Caribbean,Latin America and Caribbean,ABW,AW,Aruba,Latin America & Caribbean,NaN,High income,Not classified,Oranjestad,-70.0167,12.51670,ABW,Aruba
1,AFE,Africa Eastern and Southern,NaN,NaN,NaN,NaN,NaN,NaN,AFE,ZH,Africa Eastern and Southern,Aggregates,NaN,Aggregates,Aggregates,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,AFG,Afghanistan,Asia,Eastern Mediterranean,South Asia,Central and Southern Asia,AFG,AF,Afghanistan,South Asia,South Asia,Low income,IDA,Kabul,69.1761,34.52280,AFG,Afghanistan
3,AFW,Africa Western and Central,NaN,NaN,NaN,NaN,NaN,NaN,AFW,ZI,Africa Western and Central,Aggregates,NaN,Aggregates,Aggregates,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,AGO,Angola,Africa,Africa,Sub-Saharan Africa,Sub-Saharan Africa,AGO,AO,Angola,Sub-Saharan Africa,Sub-Saharan Africa (excluding high income),Lower middle income,IBRD,Luanda,13.2420,-8.81155,AGO,Angola
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,NaN,Wake Island,NaN,Wake Island,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
392,NaN,Non-OECD,NaN,Non-OECD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
393,NaN,Oceania,NaN,Oceania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,NaN,Asia,NaN,Asia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
